In [1]:
import pandas as pd
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

In [2]:
DIR_TABLAS = "./../tablas"

TABLAS = {
    "region":                pd.read_csv(f"{DIR_TABLAS}/region.csv"),
    "provincia":             pd.read_csv(f"{DIR_TABLAS}/provincia.csv"),
    "comuna":                pd.read_csv(f"{DIR_TABLAS}/comuna.csv"),
    "nivel_educacional":     pd.read_csv(f"{DIR_TABLAS}/nivel_educacional.csv"),
    "campo_estudio":         pd.read_csv(f"{DIR_TABLAS}/campo_estudio.csv"),
    "subcampo_estudio":      pd.read_csv(f"{DIR_TABLAS}/subcampo_estudio.csv"),
    "categoria_ocupacional": pd.read_csv(f"{DIR_TABLAS}/categoria_ocupacional.csv"),
    "rama_economica":        pd.read_csv(f"{DIR_TABLAS}/rama_economica.csv"),
    "subrama_economica":     pd.read_csv(f"{DIR_TABLAS}/subrama_economica.csv"),
    "hogar":                 pd.read_csv(f"{DIR_TABLAS}/hogar.csv"),
    "persona":               pd.read_csv(f"{DIR_TABLAS}/persona.csv"),
}

# Atajos para uso directo
df_region       = TABLAS["region"]
df_provincia    = TABLAS["provincia"]
df_comuna       = TABLAS["comuna"]
df_nivel_educ   = TABLAS["nivel_educacional"]
df_campo        = TABLAS["campo_estudio"]
df_subcampo     = TABLAS["subcampo_estudio"]
df_cat_ocup     = TABLAS["categoria_ocupacional"]
df_rama         = TABLAS["rama_economica"]
df_subrama      = TABLAS["subrama_economica"]
df_hogar        = TABLAS["hogar"]
df_persona      = TABLAS["persona"]

print("Tablas cargadas correctamente.")

Tablas cargadas correctamente.


# Tamaño en disco y filas/columnas por tabla

In [3]:
inventario = []
for nombre, df in TABLAS.items():
    ruta = f"{DIR_TABLAS}/{nombre}.csv"
    tamaño_bytes = os.path.getsize(ruta)
    if tamaño_bytes >= 1_000_000:
        tamaño = f"{tamaño_bytes/1_000_000:.2f} MB"
    elif tamaño_bytes >= 1_000:
        tamaño = f"{tamaño_bytes/1_000:.1f} KB"
    else:
        tamaño = f"{tamaño_bytes} B"

    inventario.append({
        "tabla": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "tamaño": tamaño,
    })

df_inventario = pd.DataFrame(inventario)
df_inventario

,tabla,filas,columnas,tamaño
0,region,16,2,333 B
1,provincia,56,3,925 B
2,comuna,346,3,6.5 KB
3,nivel_educacional,15,2,570 B
4,campo_estudio,11,2,414 B
5,subcampo_estudio,27,3,786 B
6,categoria_ocupacional,9,2,372 B
7,rama_economica,21,2,1.2 KB
8,subrama_economica,121,3,8.0 KB
9,hogar,78654,6,2.14 MB


# Llaves primarias y foráneas

In [4]:
ESQUEMA = pd.DataFrame([
    # tabla,                   PK,                         FKs
    ("region",                 "cod_region",               "—"),
    ("provincia",              "cod_provincia",            "cod_region → region"),
    ("comuna",                 "cod_comuna",               "cod_provincia → provincia"),
    ("nivel_educacional",      "codigo",                   "—"),
    ("campo_estudio",          "codigo",                   "—"),
    ("subcampo_estudio",       "codigo",                   "codigo_campo → campo_estudio"),
    ("categoria_ocupacional",  "codigo",                   "—"),
    ("rama_economica",         "codigo",                   "—"),
    ("subrama_economica",      "codigo",                   "codigo_rama → rama_economica"),
    ("hogar",                  "folio",                    "cod_comuna → comuna"),
    ("persona",                "(folio, id_persona)",      "folio → hogar; codigo_nivel → nivel_educacional; "
                                                            "codigo_subcampo → subcampo_estudio; "
                                                            "codigo_cat_ocup → categoria_ocupacional; "
                                                            "codigo_subrama → subrama_economica"),
], columns=["tabla", "PK", "FKs"])

ESQUEMA

,tabla,PK,FKs
0,region,cod_region,—
1,provincia,cod_provincia,cod_region → region
2,comuna,cod_comuna,cod_provincia → provincia
3,nivel_educacional,codigo,—
4,campo_estudio,codigo,—
5,subcampo_estudio,codigo,codigo_campo → campo_estudio
6,categoria_ocupacional,codigo,—
7,rama_economica,codigo,—
8,subrama_economica,codigo,codigo_rama → rama_economica
9,hogar,folio,cod_comuna → comuna


In [5]:
def inspeccionar(nombre, df):
    print(f"\n{'='*70}")
    print(f"  TABLA: {nombre}   ({len(df):,} filas × {df.shape[1]} columnas)")
    print('='*70)
    print("\n— HEAD —")
    print(df.head().to_string())
    print("\n— METADATOS —")
    meta = pd.DataFrame({
        "dtype":   df.dtypes.astype(str),
        "% nulos": (df.isna().mean() * 100).round(2),
        "únicos":  df.nunique(dropna=True),
    })
    print(meta.to_string())

In [6]:
inspeccionar("region", df_region)


  TABLA: region   (16 filas × 2 columnas)

— HEAD —
   cod_region       nombre
0           1     Tarapacá
1           2  Antofagasta
2           3      Atacama
3           4     Coquimbo
4           5   Valparaíso

— METADATOS —
            dtype  % nulos  únicos
cod_region  int64      0.0      16
nombre        str      0.0      16


In [7]:
inspeccionar("provincia", df_provincia)


  TABLA: provincia   (56 filas × 3 columnas)

— HEAD —
   cod_provincia  cod_region       nombre
0             11           1      Iquique
1             14           1    Tamarugal
2             21           2  Antofagasta
3             22           2       El Loa
4             23           2    Tocopilla

— METADATOS —
               dtype  % nulos  únicos
cod_provincia  int64      0.0      56
cod_region     int64      0.0      16
nombre           str      0.0      56


In [8]:
inspeccionar("comuna", df_comuna)


  TABLA: comuna   (346 filas × 3 columnas)

— HEAD —
   cod_comuna  cod_provincia         nombre
0        1101             11        Iquique
1        1107             11  Alto Hospicio
2        1401             14   Pozo Almonte
3        1402             14         Camiña
4        1403             14       Colchane

— METADATOS —
               dtype  % nulos  únicos
cod_comuna     int64      0.0     346
cod_provincia  int64      0.0      56
nombre           str      0.0     346


In [9]:
inspeccionar("nivel_educacional", df_nivel_educ)


  TABLA: nivel_educacional   (15 filas × 2 columnas)

— HEAD —
   codigo                                               descripcion
0       1                                             Nunca asistió
1       2                                                 Sala cuna
2       3               Jardín Infantil (Medio menor y Medio mayor)
3       4  Prekínder / Kínder (Transición menor y Transición Mayor)
4       5                          Educación Especial (Diferencial)

— METADATOS —
             dtype  % nulos  únicos
codigo       int64      0.0      15
descripcion    str      0.0      15


In [10]:
inspeccionar("campo_estudio", df_campo)


  TABLA: campo_estudio   (11 filas × 2 columnas)

— HEAD —
   codigo                           descripcion
0       1                     Salud y Bienestar
1       2  Ingeniería, Industria y Construcción
2       3                             Educación
3       4                             Servicios
4       5  Administración de Empresas y Derecho

— METADATOS —
             dtype  % nulos  únicos
codigo       int64      0.0      11
descripcion    str      0.0      11


In [11]:
inspeccionar("subcampo_estudio", df_subcampo)


  TABLA: subcampo_estudio   (27 filas × 3 columnas)

— HEAD —
   codigo  codigo_campo                      descripcion
0       1             1                            Salud
1      15             1                        Bienestar
2       2             2  Ingeniería y Profesiones Afines
3      11             2      Arquitectura y Construcción
4      23             2           Industria y Producción

— METADATOS —
              dtype  % nulos  únicos
codigo        int64      0.0      27
codigo_campo  int64      0.0      10
descripcion     str      0.0      27


In [12]:
inspeccionar("categoria_ocupacional", df_cat_ocup)


  TABLA: categoria_ocupacional   (9 filas × 2 columnas)

— HEAD —
   codigo                                                                descripcion
0       1                                                   Patrón(a) o empleador(a)
1       2                                            Trabajador(a) por cuenta propia
2       3  Empleado(a) u obrero(a) del sector público (Gobierno Central o Municipal)
3       4                               Empleado(a) u obrero(a) de empresas públicas
4       5                                 Empleado(a) u obrero(a) del sector privado

— METADATOS —
             dtype  % nulos  únicos
codigo       int64      0.0       9
descripcion    str      0.0       9


In [13]:
inspeccionar("rama_economica", df_rama)


  TABLA: rama_economica   (21 filas × 2 columnas)

— HEAD —
   codigo                                                                                 descripcion
0       1                                                Agricultura, ganadería, silvicultura y pesca
1       2                                                             Explotación de minas y canteras
2       3                                                                   Industrias manufactureras
3       4                                 Suministro de electricidad, gas, vapor y aire acondicionado
4       5  Suministro de agua; evacuación de aguas residuales, gestión de desechos y descontaminación

— METADATOS —
             dtype  % nulos  únicos
codigo       int64      0.0      21
descripcion    str      0.0      21


In [14]:
inspeccionar("subrama_economica", df_subrama)


  TABLA: subrama_economica   (121 filas × 3 columnas)

— HEAD —
   codigo  codigo_rama                                                                                       descripcion
0     101            1                                                                    Cultivo de plantas no perennes
1     102            1                                                                       Cultivo de plantas perennes
2     103            1                                                                                         Ganadería
3     199            1  Otras actividades relacionadas con la agricultura, ganadería, caza y actividades de apoyo n.c.p.
4     201            1                             Silvicultura, extracción de madera y actividades de servicios conexas

— METADATOS —
             dtype  % nulos  únicos
codigo       int64      0.0     121
codigo_rama  int64      0.0      21
descripcion    str      0.0     121


In [15]:
inspeccionar("hogar", df_hogar)


  TABLA: hogar   (78,654 filas × 6 columnas)

— HEAD —
       folio  area  expr  expp  expc  cod_comuna
0  100020301     1   264   285   371       13127
1  100020401     1   295   286   371       13127
2  100020501     1   373   285   371       13127
3  100020801     1   266   285   372       13127
4  100070201     1    72    60    82        2203

— METADATOS —
            dtype  % nulos  únicos
folio       int64      0.0   78654
area        int64      0.0       2
expr        int64      0.0     687
expp        int64      0.0     485
expc        int64      0.0     498
cod_comuna  int64      0.0     335


In [16]:
inspeccionar("persona", df_persona)


  TABLA: persona   (218,367 filas × 10 columnas)

— HEAD —
       folio  id_persona  sexo  edad  activ  codigo_nivel  codigo_subcampo  codigo_cat_ocup  codigo_subrama    yoprcor
0  100020301           1     1    62    3.0           9.0              NaN              NaN             NaN        NaN
1  100020301           2     2    92    3.0           7.0              NaN              NaN             NaN        NaN
2  100020401           1     1    61    1.0           7.0              NaN              5.0          4805.0  1200000.0
3  100020401           2     2    61    1.0           7.0              NaN              5.0          6801.0   500000.0
4  100020401           3     1    32    1.0          12.0              5.0              5.0          4808.0   700000.0

— METADATOS —
                   dtype  % nulos  únicos
folio              int64     0.00   78654
id_persona         int64     0.00      22
sexo               int64     0.00       2
edad               int64     0.00     107
a

# Rangos de valores numéricos

## columnas numéricas en persona y hogar

In [17]:
cols_numericas = {
    "persona": ["edad", "yoprcor"],
    "hogar":   ["expr", "expp", "expc"],
}

filas = []
for tabla, cols in cols_numericas.items():
    df = TABLAS[tabla]
    for col in cols:
        serie = df[col].dropna()
        filas.append({
            "tabla":    tabla,
            "columna":  col,
            "min":      serie.min(),
            "p25":      serie.quantile(0.25),
            "mediana":  serie.median(),
            "p75":      serie.quantile(0.75),
            "max":      serie.max(),
            "no_nulos": int(serie.count()),
        })

pd.DataFrame(filas)

,tabla,columna,min,p25,mediana,p75,max,no_nulos
0,persona,edad,0.0,20.0,39.0,59.0,107.0,218367
1,persona,yoprcor,4000.0,410000.0,540000.0,800000.0,20000000.0,92548
2,hogar,expr,1.0,43.0,67.0,108.0,2060.0,78654
3,hogar,expp,1.0,47.0,71.0,101.0,1544.0,78654
4,hogar,expc,1.0,46.0,72.0,102.0,1459.0,78654


## columnas categóricas de persona

In [18]:
cols_cat_persona = [
    "sexo", "activ",
    "codigo_nivel", "codigo_subcampo",
    "codigo_cat_ocup", "codigo_subrama",
]

resumen_cat = pd.DataFrame({
    "columna":      cols_cat_persona,
    "valores_unicos": [df_persona[c].nunique(dropna=True) for c in cols_cat_persona],
    "% nulos":      [round(df_persona[c].isna().mean() * 100, 2) for c in cols_cat_persona],
})
resumen_cat

,columna,valores_unicos,% nulos
0,sexo,2,0.00
1,activ,3,16.89
2,codigo_nivel,15,0.00
3,codigo_subcampo,27,72.10
4,codigo_cat_ocup,9,56.30
5,codigo_subrama,121,56.31


# Cardinalidad de las relaciones

In [20]:
print("Geografía:")
print(f"  Provincias por región (promedio):  {len(df_provincia) / len(df_region):.1f}")
print(f"  Comunas por provincia (promedio):  {len(df_comuna) / len(df_provincia):.1f}")
print(f"  Comunas por región (promedio):     {len(df_comuna) / len(df_region):.1f}")

print("\nEducación:")
print(f"  Subcampos por campo (promedio):    {len(df_subcampo) / len(df_campo):.1f}")

print("\nActividad económica:")
print(f"  Subramas por rama (promedio):      {len(df_subrama) / len(df_rama):.1f}")

print("\nUnidad de análisis:")
print(f"  Personas por hogar (promedio):     {len(df_persona) / len(df_hogar):.2f}")
print(f"  Hogares por comuna (promedio):     {len(df_hogar) / len(df_comuna):.1f}")

Geografía:
  Provincias por región (promedio):  3.5
  Comunas por provincia (promedio):  6.2
  Comunas por región (promedio):     21.6

Educación:
  Subcampos por campo (promedio):    2.5

Actividad económica:
  Subramas por rama (promedio):      5.8

Unidad de análisis:
  Personas por hogar (promedio):     2.78
  Hogares por comuna (promedio):     227.3


## Personas por hogar

In [21]:
personas_por_hogar = df_persona.groupby("folio").size()

print("Distribución de personas por hogar:")
print(personas_por_hogar.describe().round(2))

print("\nDistribución por tamaño:")
print(personas_por_hogar.value_counts().sort_index().rename("hogares").to_string())

Distribución de personas por hogar:
count    78654.00
mean         2.78
std          1.46
min          1.00
25%          2.00
50%          3.00
75%          4.00
max         19.00
dtype: float64

Distribución por tamaño:
1     15910
2     22783
3     17803
4     12855
5      5888
6      2187
7       725
8       280
9       140
10       41
11       20
12        8
13        9
14        1
16        2
17        1
19        1


# queries típicas

## personas por región

In [22]:
# pandas
q1 = (
    df_persona
    .merge(df_hogar[["folio", "cod_comuna"]], on="folio")
    .merge(df_comuna[["cod_comuna", "cod_provincia"]], on="cod_comuna")
    .merge(df_provincia[["cod_provincia", "cod_region"]], on="cod_provincia")
    .merge(df_region, on="cod_region")
    .groupby("nombre")
    .size()
    .sort_values(ascending=False)
    .rename("personas")
    .reset_index()
)
q1.head(10)

# Equivalente SQL:
# SELECT r.nombre, COUNT(*) AS personas
# FROM persona p
# JOIN hogar h     ON p.folio = h.folio
# JOIN comuna c    ON h.cod_comuna = c.cod_comuna
# JOIN provincia pr ON c.cod_provincia = pr.cod_provincia
# JOIN region r    ON pr.cod_region = r.cod_region
# GROUP BY r.nombre
# ORDER BY personas DESC;

,nombre,personas
0,Región Metropolitana de Santiago,40658
1,Valparaíso,22951
2,Biobío,22447
3,Maule,14657
4,Libertador Gral. Bernardo O'Higgins,14590
5,La Araucanía,14457
6,Los Lagos,11343
7,Antofagasta,10605
8,Coquimbo,10285
9,Tarapacá,9732


## ingreso promedio de personas ocupadas por rama económica

In [24]:
# pandas
q2 = (
    df_persona[df_persona["activ"] == 1]
    .merge(
        df_subrama[["codigo", "codigo_rama"]].rename(columns={"codigo": "codigo_subrama"}),
        on="codigo_subrama",
    )
    .merge(
        df_rama.rename(columns={"codigo": "codigo_rama", "descripcion": "rama"}),
        on="codigo_rama",
    )
    .groupby("rama")["yoprcor"]
    .mean()
    .sort_values(ascending=False)
    .round(0)
    .rename("ingreso_promedio")
    .reset_index()
)
q2.head(10)

# Equivalente SQL:
# SELECT re.descripcion AS rama, AVG(p.yoprcor) AS ingreso_promedio
# FROM persona p
# JOIN subrama_economica sre ON p.codigo_subrama = sre.codigo
# JOIN rama_economica re     ON sre.codigo_rama = re.codigo
# WHERE p.activ = 1
# GROUP BY re.descripcion
# ORDER BY ingreso_promedio DESC;

,rama,ingreso_promedio
0,Actividades de organizaciones y órganos extraterritoriales,2341615.0
1,Actividades financieras y de seguros,1466092.0
2,Explotación de minas y canteras,1262549.0
3,Información y comunicaciones,1249888.0
4,"Actividades profesionales, científicas y técnicas",1212976.0
5,"Suministro de electricidad, gas, vapor y aire acondicionado",1212188.0
6,Administración pública y defensa; planes de seguridad social de afiliación o...,1127866.0
7,Actividades de atención de la salud humana y de asistencia social,1021444.0
8,Actividades inmobiliarias,991357.0
9,Enseñanza,880526.0


## distribución educacional por sexo (adultos 25+)

In [25]:
# pandas
q3 = (
    df_persona[df_persona["edad"] >= 25]
    .merge(df_nivel_educ, left_on="codigo_nivel", right_on="codigo")
    .groupby(["descripcion", "sexo"])
    .size()
    .unstack("sexo", fill_value=0)
    .rename(columns={1: "hombres", 2: "mujeres"})
)
q3

# Equivalente SQL (necesita pivot manual o usar FILTER):
# SELECT ne.descripcion,
#        SUM(CASE WHEN p.sexo = 1 THEN 1 ELSE 0 END) AS hombres,
#        SUM(CASE WHEN p.sexo = 2 THEN 1 ELSE 0 END) AS mujeres
# FROM persona p
# JOIN nivel_educacional ne ON p.codigo_nivel = ne.codigo
# WHERE p.edad >= 25
# GROUP BY ne.descripcion;

sexo,hombres,mujeres
descripcion,,
Doctorado,222,160
Educación Básica,12827,14583
Educación Especial (Diferencial),379,261
Educación Media Científico-Humanista,20076,22725
Educación Media Técnica Profesional,7234,7628
Humanidades (Sistema Antiguo),1885,2942
Jardín Infantil (Medio menor y Medio mayor),1,4
Magíster o maestría,1193,1262
Nunca asistió,1004,1415


## comunas con mayor tasa de ocupación (top 10)

In [26]:
# pandas
ocup_por_comuna = (
    df_persona[df_persona["edad"] >= 15]
    .merge(df_hogar[["folio", "cod_comuna"]], on="folio")
    .groupby("cod_comuna")
    .agg(
        total_15mas=("activ", "size"),
        ocupados=("activ", lambda s: (s == 1).sum()),
    )
)
ocup_por_comuna["tasa_ocupacion"] = (
    ocup_por_comuna["ocupados"] / ocup_por_comuna["total_15mas"] * 100
).round(1)

q4 = (
    ocup_por_comuna
    .merge(df_comuna[["cod_comuna", "nombre"]], on="cod_comuna")
    .sort_values("tasa_ocupacion", ascending=False)
    .head(10)
)
q4

# Equivalente SQL:
# SELECT c.nombre,
#        COUNT(*) AS total_15mas,
#        SUM(CASE WHEN p.activ = 1 THEN 1 ELSE 0 END) AS ocupados,
#        100.0 * SUM(CASE WHEN p.activ = 1 THEN 1 ELSE 0 END) / COUNT(*) AS tasa_ocupacion
# FROM persona p
# JOIN hogar h  ON p.folio = h.folio
# JOIN comuna c ON h.cod_comuna = c.cod_comuna
# WHERE p.edad >= 15
# GROUP BY c.nombre
# ORDER BY tasa_ocupacion DESC
# LIMIT 10;

,cod_comuna,total_15mas,ocupados,tasa_ocupacion,nombre
240,12201,30,28,93.3,Cabo de Hornos
233,11303,40,37,92.5,Tortel
238,12103,19,17,89.5,Río Verde
239,12104,24,20,83.3,San Gregorio
245,12402,19,15,78.9,Torres del Paine
243,12303,31,24,77.4,Timaukel
313,15202,16,12,75.0,General Lagos
246,13101,2018,1467,72.7,Santiago
242,12302,32,23,71.9,Primavera
237,12102,21,15,71.4,Laguna Blanca


NOTAS DE USO

1. FACTORES DE EXPANSIÓN
   - Los conteos crudos NO son representativos de la población.
   - Para estimar valores poblacionales:
       * Generalización nacional/regional: usar expr (de hogar)
       * Análisis a nivel provincial:        usar expp
       * Análisis a nivel comunal:           usar expc
   - Ejemplo pandas: df.groupby('x')['expr'].sum() en vez de groupby().size()

2. COLUMNAS NULAS
   - activ es NULL para personas < 15 años (no aplica).
   - o15, rama1, rama4 son NULL para personas no ocupadas.
   - yoprcor es NULL para personas sin ingreso de ocupación.
   - cinef13_area/subarea son NULL para personas sin educación superior (~72%).
   - codigo_subcampo en Persona puede ser NULL legítimamente.

3. CÓDIGOS ESPECIALES (-99, -88, -66)
   - Ya fueron reemplazados por NaN durante el procesamiento.
   - Si encuentras alguno en los CSV, revisar el pipeline de limpieza.

4. CONVENCIONES DE CASEN
   - sexo:  1 = Hombre, 2 = Mujer
   - area:  1 = Urbano, 2 = Rural
   - activ: 1 = Ocupado, 2 = Desocupado, 3 = Inactivo

5. JERARQUÍAS GEOGRÁFICAS (los códigos NO son consecutivos)
   - cod_region:    1-16  (no incluye 13.5, etc.)
   - cod_provincia: 11-163 (primer(os) dígito(s) = región)
   - cod_comuna:    1101-16305 (primeros 2-3 dígitos = provincia)

6. PERSONAS SIN INGRESO REGISTRADO
   - ~57.6% de las personas tienen yoprcor NULL.
   - Esto NO significa que no tengan ingresos: significa que no tienen
     "ingreso de ocupación principal" (no son ocupados, o no aplica).
   - Para analizar ingresos, filtrar: WHERE yoprcor IS NOT NULL